# Урок 08 - Патерн Багатоагентного Проектування


## Налаштування


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Чому багатофакторні системи?

Завдання реального світу, як-от планування поїздок, включають багато різних видів експертизи — логістику, місцеві знання, бюджетування тощо. Один агент, що намагається впоратися з усім, швидко стає незручним.

Багатофакторні системи вирішують це за допомогою **спеціалізації**: кожен агент зосереджується на одній сфері експертизи, даючи результат вищої якості, ніж універсаліст. Вони також покращують **масштабованість** — ви можете додати нових агентів (наприклад, спеціаліста з авіаперельотів, критика ресторанів) без переписування існуючого робочого процесу. Агенти працюють разом через структурований конвеєр, передаючи контекст від одного до іншого.


## Створення спеціалізованих агентів


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Побудова послідовного робочого процесу

`WorkflowBuilder` дозволяє побудувати агентів у вигляді орієнтованого графа. Тут ми створюємо простий двокроковий конвеєр: **TravelPlanner** складає маршрут, а потім **TravelConcierge** переглядає та вдосконалює його.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавання більше агентів до робочого процесу

Однією з найбільших переваг багатоагентної схеми є те, наскільки легко її розширити. Нижче ми додаємо агента **BudgetReviewer**, який перевіряє план відповідно до бюджету мандрівника, позначає позиції, які можуть перевищити ліміт витрат, і пропонує альтернативи для економії грошей. Тепер робочий процес виконує три агенти послідовно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Підсумок

У цьому уроці ви навчилися:

1. **Створювати спеціалізованих агентів** — кожен із зосередженою роллю (планування, консьєрж, перегляд бюджету).
2. **Об'єднувати агентів у послідовний робочий процес** за допомогою `WorkflowBuilder` і `add_edge`.
3. **Потоково передавати вихідні дані** з багатоагентної конвеєрної лінії, відстежуючи, який агент говорить.
4. **Розширювати робочий процес**, додаючи нових агентів у ланцюг без зміни існуючих.

Шаблон дизайну з багатьма агентами зберігає кожного агента простим, одночасно створюючи більш насичені, детально переглянуті результати, ніж будь-який один агент міг би досягти сам.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
